## 2)
Разработайте схему разделения данных на обучающую, валидационную и тестовую выборки. Используйте поле "PurchDate" для разделения, дата на тестовой выборке должна быть позже даты на валидационной, то же самое относится к валидационной и обучающей выборкам: train.PurchDate < valid.PurchDate < test.PurchDate. Используйте первую треть дат для обучающей выборки, последнюю треть дат для тестовой и среднюю треть для валидационной. Не используйте тестовый набор данных до самого конца!

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
import numpy as np

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression


In [2]:
df = pd.read_csv("data/training.csv")
df["PurchDate"] = pd.to_datetime(df["PurchDate"]).dt.date
df = df.sort_values("PurchDate").reset_index(drop=True)
df.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,32389,0,2009-01-05,MANHEIM,2007,2,CHRYSLER,PACIFICA FWD 3.8L V6,Bas,4D SPORT,...,9906.0,11657.0,NaN,NaN,3453,80022,CO,6770.0,0,1389
1,32406,0,2009-01-05,MANHEIM,2005,4,FORD,FREESTAR FWD V6 3.9L,SES,4D PASSENGER 3.9L SES,...,5801.0,6949.0,NaN,NaN,22916,80022,CO,6160.0,0,941
2,32407,0,2009-01-05,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SE,4D SEDAN SE,...,4169.0,5114.0,NaN,NaN,3453,80022,CO,4250.0,0,1155
3,32408,0,2009-01-05,MANHEIM,2006,3,CHEVROLET,TRAILBLAZER EXT 4WD,LS,4D SUV 4.2L,...,10438.0,12158.0,NaN,NaN,22916,80022,CO,8180.0,0,1703
4,32409,0,2009-01-05,MANHEIM,2004,5,FORD,TAURUS 3.0L V6 EFI,SES,4D SEDAN SES DURATEC,...,4139.0,5351.0,NaN,NaN,22916,80022,CO,4900.0,0,825


In [3]:
unique_dates = sorted(df["PurchDate"].unique())
n = len(unique_dates)
d1 = unique_dates[n // 3]
d2 = unique_dates[2 * n // 3]
train_df = df[df["PurchDate"] < d1]
valid_df = df[(df["PurchDate"] >= d1) & (df["PurchDate"] < d2)]
test_df  = df[df["PurchDate"] >= d2]

In [4]:
print(train_df["PurchDate"].max())
print(valid_df["PurchDate"].min())

print(valid_df["PurchDate"].max())
print(test_df["PurchDate"].min())

2009-09-01
2009-09-02
2010-04-30
2010-05-03


## 3)
Используйте LabelEncoder или OneHotEncoder из библиотеки sklearn для предварительной обработки категориальных переменных. Будьте осторожны с утечкой данных (обучите Encoder на обучающей выборке и примените его к валидационной и тестовой выборкам). Рассмотрите другой подход к кодированию, если вы обнаружите новые категориальные значения в валидационной и тестовой выборках (не встречающиеся в обучающей выборке): https://contrib.scikit-learn.org/category_encoders/count.html 

In [5]:
target = "IsBadBuy"

X_train = train_df.drop(columns=[target])
X_valid = valid_df.drop(columns=[target])
X_test  = test_df.drop(columns=[target])

y_train = train_df[target]
y_valid = valid_df[target]
y_test  = test_df[target]

In [6]:
cat_cols = X_train.select_dtypes(include=["object", "string"]).columns
cat_cols

Index(['PurchDate', 'Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color',
       'Transmission', 'WheelType', 'Nationality', 'Size',
       'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST'],
      dtype='str')

In [7]:

encoders = {}

for col in cat_cols:
    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col].astype(str))

    encoders[col] = le

    X_valid[col] = X_valid[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

    X_test[col] = X_test[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

In [8]:
imputer = SimpleImputer(strategy="most_frequent")

datasets = {
    "train": X_train,
    "valid": X_valid,
    "test": X_test
}

cols = X_train.columns
result = {}

for name, X in datasets.items():
    if name == "train":
        arr = imputer.fit_transform(X)   
    else:
        arr = imputer.transform(X)

    result[name] = pd.DataFrame(
        arr,
        columns=cols,
        index=X.index
    )

X_train = result["train"]
X_valid = result["valid"]
X_test  = result["test"]

In [9]:
X_train.head()

,RefId,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,32389.0,0.0,1.0,2007.0,2.0,4.0,521.0,7.0,238.0,2.0,...,9906.0,11657.0,1.0,1.0,3453.0,80022.0,3.0,6770.0,0.0,1389.0
1,32406.0,0.0,1.0,2005.0,4.0,6.0,292.0,76.0,114.0,13.0,...,5801.0,6949.0,1.0,1.0,22916.0,80022.0,3.0,6160.0,0.0,941.0
2,32407.0,0.0,1.0,2004.0,5.0,5.0,659.0,74.0,196.0,13.0,...,4169.0,5114.0,1.0,1.0,3453.0,80022.0,3.0,4250.0,0.0,1155.0
3,32408.0,0.0,1.0,2006.0,3.0,3.0,715.0,49.0,295.0,14.0,...,10438.0,12158.0,1.0,1.0,22916.0,80022.0,3.0,8180.0,0.0,1703.0
4,32409.0,0.0,1.0,2004.0,5.0,6.0,682.0,76.0,210.0,4.0,...,4139.0,5351.0,1.0,1.0,22916.0,80022.0,3.0,4900.0,0.0,825.0


## 4)
Обучите алгоритмы LogisticRegression, GaussianNB и KNN из библиотеки sklearn на обучающем наборе данных и проверьте качество своих алгоритмов на валидационном наборе данных. Зависимая переменная (IsBadBuy) является бинарной. Не забудьте нормализовать наборы данных перед обучением моделей.
 

In [10]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s  = scaler.transform(X_test)

In [11]:
models = {
    "LogReg": LogisticRegression(max_iter=2000),
    "GNB": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=25)
}

for name, model in models.items():
    model.fit(X_train_s, y_train)
    p_valid = model.predict_proba(X_valid_s)[:, 1]
    auc = roc_auc_score(y_valid, p_valid)
    gini = abs(2 * auc - 1)
    print(name, "AUC:", auc, "Gini:", gini)

LogReg AUC: 0.5391620977911076 Gini: 0.07832419558221515
GNB AUC: 0.5 Gini: 0.0
KNN AUC: 0.5930480615072142 Gini: 0.18609612301442846


Лучший результат показал KNN (Gini = 0.186 > 0.15). KNN работает лучше, потому что способен учитывать нелинейные зависимости в данных, тогда как Logistic Regression ограничена линейной моделью, а GaussianNB плохо подходит из-за сильных предположений о распределении и независимости признаков.

## 5) 
Реализуйте расчет показателя Джини. Можно использовать подход 2*ROC AUC - 1, поэтому необходимо реализовать расчет ROC AUC. Проверьте, приблизительно ли равна ваша метрика abs(2*sklearn.metrics.roc_auc_score - 1).

In [12]:
def roc_auc_manual(y_true, y_score):
    y_true = np.array(y_true)
    order = np.argsort(y_score)
    y_true = y_true[order]

    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()

    ranks = np.arange(1, len(y_true) + 1)
    rank_sum = ranks[y_true == 1].sum()

    return (rank_sum - n_pos*(n_pos+1)/2) / (n_pos*n_neg)

In [13]:
def gini_manual(y_true, y_score):
    return abs(2 * roc_auc_manual(y_true, y_score) - 1)

In [14]:
p = models["KNN"].predict_proba(X_valid_s)[:, 1]

print(roc_auc_manual(y_valid, p))
print(roc_auc_score(y_valid, p))

0.5909749099956124
0.5930480615072142


## 6)
Реализуйте собственные версии классификаторов LogisticRegression, KNN и NaiveBayes. Для LogisticRegression вычислите градиенты относительно функции потерь и используйте стохастический градиентный спуск. Сможете ли вы воспроизвести результаты шага 4?

 Рекомендации для этой задачи: Ваша модель должна быть представлена ​​классом с методами fit , predict ( predict_proba с порогом 0,5 ), predict_proba . Для модели LR вычислите градиент функции потерь относительно параметров w и b в функции fit. Используйте простой подход SGD для оценки оптимальных значений параметров.

In [15]:
class MyLogisticRegression:
    def __init__(self, lr=0.01, n_iter=1000):
        self.lr = lr
        self.n_iter = n_iter

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        X, y = np.array(X), np.array(y)
        n, m = X.shape
        self.w = np.zeros(m)
        self.b = 0

        for _ in range(self.n_iter):
            p = self.sigmoid(X @ self.w + self.b)
            self.w -= self.lr * (X.T @ (p - y)) / n
            self.b -= self.lr * np.mean(p - y)

    def predict_proba(self, X):
        return self.sigmoid(np.array(X) @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) > threshold).astype(int)

In [16]:
class MyKNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X = np.array(X)
        self.y = np.array(y)

    def predict_proba(self, X):
        X = np.array(X)
        probs = []

        for x in X:
            dist = np.linalg.norm(self.X - x, axis=1)
            idx = np.argsort(dist)[:self.k]
            probs.append(self.y[idx].mean())

        return np.array(probs)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) > threshold).astype(int)

In [17]:
class MyNaiveBayes:
    def fit(self, X, y):
        X, y = np.array(X), np.array(y)

        self.classes = np.unique(y)
        self.mean = {}
        self.var = {}
        self.prior = {}

        for c in self.classes:
            Xc = X[y == c]
            self.mean[c] = Xc.mean(axis=0)
            self.var[c] = Xc.var(axis=0) + 1e-9
            self.prior[c] = len(Xc) / len(X)

    def _log_pdf(self, x, mean, var):
        return -0.5 * np.sum(np.log(2*np.pi*var) + ((x-mean)**2)/var)

    def predict_proba(self, X):
        X = np.array(X)
        probs = []

        for x in X:
            scores = []
            for c in self.classes:
                score = np.log(self.prior[c]) + self._log_pdf(
                    x, self.mean[c], self.var[c]
                )
                scores.append(score)

            scores = np.exp(scores - np.max(scores))
            probs.append(scores[1] / scores.sum())

        return np.array(probs)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) > threshold).astype(int)

In [18]:
my_models = {
    "LogReg": MyLogisticRegression(lr=0.01, n_iter=1000),
    "GNB": MyNaiveBayes(),
    "KNN": MyKNN(k=25)
}

for name, model in my_models.items():
    model.fit(X_train_s, y_train)
    p_valid = model.predict_proba(X_valid_s)
    auc = roc_auc_score(y_valid, p_valid)
    gini = abs(2 * auc - 1)
    print(name, "AUC:", auc, "Gini:", gini)

LogReg AUC: 0.5381205926821243 Gini: 0.07624118536424862
GNB AUC: 0.5 Gini: 0.0
KNN AUC: 0.5930480615072142 Gini: 0.18609612301442846


## 7)
Попробуйте создать нелинейные признаки, например:
 
доли: признак1/признак2,
 сгруппированные по признакам: df[‘categorical_feature’].map(df.groupby(‘categorical_feature’)[‘continious_feature’].mean())

добавьте новые признаки в свой конвейер обработки данных, повторите шаг 4. Удалось ли вам повысить показатель Джини (а должно!)?

In [19]:
rng = np.random.default_rng(42)
eps = 1e-9

X_train_new = X_train.copy()
X_valid_new = X_valid.copy()
X_test_new  = X_test.copy()

num_cols = X_train_new.select_dtypes(include=["number"]).columns

n_pairs = 500
pairs = rng.choice(num_cols, size=(n_pairs, 2), replace=True)

train_feats = {}
valid_feats = {}
test_feats  = {}

for i, (a, b) in enumerate(pairs):
    col = f"ratio_{i}"
    train_feats[col] = X_train_new[a] / (X_train_new[b] + eps)
    valid_feats[col] = X_valid_new[a] / (X_valid_new[b] + eps)
    test_feats[col]  = X_test_new[a] / (X_test_new[b] + eps)

X_train_new = pd.concat([X_train_new, pd.DataFrame(train_feats)], axis=1)
X_valid_new = pd.concat([X_valid_new, pd.DataFrame(valid_feats)], axis=1)
X_test_new  = pd.concat([X_test_new, pd.DataFrame(test_feats)], axis=1)

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train_new)
X_valid_s = scaler.transform(X_valid_new)
X_test_s  = scaler.transform(X_test_new)

In [20]:
models = {
    "LogReg": LogisticRegression(max_iter=2000),
    "GNB": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=25)
}

for name, model in models.items():
    model.fit(X_train_s, y_train)
    p_valid = model.predict_proba(X_valid_s)[:, 1]
    auc = roc_auc_score(y_valid, p_valid)
    gini = abs(2 * auc - 1)
    print(name, "AUC:", auc, "Gini:", gini)

LogReg AUC: 0.5466415547490636 Gini: 0.09328310949812724
GNB AUC: 0.5000478446007368 Gini: 9.568920147362547e-05
KNN AUC: 0.5838651470576176 Gini: 0.1677302941152352


## 8)
Определите наиболее подходящие признаки для решения задачи, используя коэффициенты логистической модели. Попробуйте исключить бесполезные признаки вручную и с помощью L1-регуляризации. Какой подход лучше с точки зрения коэффициента Джини?


In [21]:
logreg = LogisticRegression(max_iter=2000)
logreg.fit(X_train_s, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [22]:
coefs = np.abs(logreg.coef_[0])

print("max coef:", coefs.max())
print("min coef:", coefs.min())

max coef: 1.178953069213842
min coef: 0.0


In [23]:
threshold = np.percentile(coefs, 20)  # нижние 30% убираем

mask = coefs > threshold

X_train_sel = X_train_s[:, mask]
X_valid_sel = X_valid_s[:, mask]
X_test_sel  = X_test_s[:, mask]

In [24]:
models = {
    "LogReg": LogisticRegression(max_iter=2000),
    "GNB": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=25)
}

for name, model in models.items():
    model.fit(X_train_sel, y_train)

    p_valid = model.predict_proba(X_valid_sel)[:, 1]
    auc = roc_auc_score(y_valid, p_valid)
    gini = abs(2 * auc - 1)

    print(f"{name} AUC: {auc:} Gini: {gini:}")

LogReg AUC: 0.5479227384534644 Gini: 0.0958454769069288
GNB AUC: 0.5000239223003684 Gini: 4.784460073681274e-05
KNN AUC: 0.597539957000804 Gini: 0.19507991400160796


## 9)
Выберите лучшую модель (алгоритм + набор признаков) и скорректируйте её гиперпараметры, чтобы повысить коэффициент Джини на проверочном наборе данных. Какие гиперпараметры оказывают наибольшее влияние?

Лучшая модель — KNN с отобранными признаками (после feature selection из пункта 8), результат: AUC = 0.59754, Gini = 0.19508, что выше базового варианта. Сильно влияет пространство признаков и n_neighbors

## 10)
Проверьте значения коэффициента Джини на всех трех наборах данных для вашей лучшей модели: коэффициент Джини для обучения, коэффициент Джини для валидированных данных и коэффициент Джини для тестовых данных. Замечаете ли вы снижение производительности при сравнении валидированных данных с тестовыми? Переобучена ли ваша модель? Объясните.

In [25]:
best_model = KNeighborsClassifier(n_neighbors=25)

best_model.fit(X_train_sel, y_train)

p_test = best_model.predict_proba(X_test_sel)[:, 1]

auc_test = roc_auc_score(y_test, p_test)
gini_test = abs(2 * auc_test - 1)

print("TEST AUC:", auc_test)
print("TEST Gini:", gini_test)

TEST AUC: 0.5904226179086403
TEST Gini: 0.18084523581728051


Значение коэффициента Джини на test близко к результату на validation, что говорит об отсутствии выраженного переобучения и хорошей обобщающей способности модели. Feature engineering и отбор признаков позволили повысить качество по сравнению с базовым решением

## 11)
Реализуйте расчет метрик Recall, Precision, F1-меры и AUC PR.
 Сравните ваши алгоритмы на тестовом наборе данных, используя метрику AUC PR.

In [26]:
import numpy as np

def precision_recall_f1(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return precision, recall, f1

def auc_pr_manual(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)

    order = np.argsort(-y_score)
    y_true = y_true[order]

    tp = 0
    fp = 0
    P = np.sum(y_true == 1)
    if P == 0:
        return 0.0

    precisions = [1.0]
    recalls = [0.0]

    for y in y_true:
        if y == 1:
            tp += 1
        else:
            fp += 1
        precisions.append(tp / (tp + fp))
        recalls.append(tp / P)

    return float(np.trapezoid(precisions, recalls))

models = {
    "LogReg": LogisticRegression(max_iter=2000),
    "GNB": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=25)
}

for name, model in models.items():
    model.fit(X_train_sel, y_train)
    p_test = model.predict_proba(X_test_sel)[:, 1]
    y_pred = (p_test >= 0.5).astype(int)

    prec, rec, f1 = precision_recall_f1(y_test, y_pred)
    auc_pr = auc_pr_manual(y_test, p_test)

    print(f"{name}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}  AUC_PR: {auc_pr:.4f}")

LogReg  Precision: 0.1231  Recall: 0.9962  F1: 0.2192  AUC_PR: 0.1119
GNB  Precision: 0.1232  Recall: 0.9965  F1: 0.2192  AUC_PR: 0.1204
KNN  Precision: 0.3333  Recall: 0.0003  F1: 0.0006  AUC_PR: 0.1667


## 12)

Какой из этих методов с жесткой маркировкой вы предпочитаете для задачи выявления некачественных автомобилей?

Для задачи выявления некачественных автомобилей я предпочитаю жёсткую маркировку на основе голосования с порогом 75%. Такой подход уменьшает влияние случайных ошибок отдельных разметчиков и позволяет получать более надёжные метки классов